In [ ]:
import pandas as pd
import numpy as np

# CHARGEMENT
df = pd.read_csv("D:\\rouge_file\\Analyse_du_performance_de_la_bourse_casablanca\\csv_clean\\Actions_RENAMED.csv")

# Suppression lignes parasites et doublons
df = df[df['Type'] != 'Type'].drop_duplicates().copy()

print(f'Shape apres nettoyage basique : {df.shape}')
print(f'Nulls avant traitement        : {df.isnull().sum().sum()}')
print()

# GROUPE 1 — CARACTERISTIQUES STABLES DE LA SOCIETE
# Forward Fill par société (Libelle)
# Logique : ces colonnes decrivent des infos fixes d une societe
# (son secteur, son capital, sa valeur nominale...).
# Si une valeur manque pour une seance, on prend celle
# de la seance precedente pour le meme titre.
cols_ffill = [
    'Capital en titres',
    'Val. Nom.',
    'Secteur activité',
    'Cycle de négociation',
    'Instruments',
    'Libelle',
    'Cours de référence',
    'Date_Cours_Ref',
    'Cours_Reference_Seance',
    'Meilleur_Demande',
    'Cours_Cloture',
    'Plus_Haut_Janv',
    'Plus_Bas_Janv',
    'Meilleur_Offre',
]

# CORRECTION LIBELLE MANQUANT
# Logique : quand Libelle est null, le nom de la societe est
# encode dans la colonne Instruments sous la forme :
# "MA0000012528 TGCC S.A" — on extrait la partie apres le code ISIN
mask = df['Libelle'].isnull() & df['Instruments'].notnull()
df.loc[mask, 'Libelle'] = df.loc[mask, 'Instruments'].str.replace(r'^MA\d+\s*', '', regex=True).str.strip()
print(f'Libelles extraits depuis Instruments : {mask.sum()}')

# On trie par Libelle pour que le ffill soit coherent par societe
df = df.sort_values(['Libelle', 'Date_Cours_Ref'])

for col in cols_ffill:
    if col in df.columns:
        avant = df[col].isnull().sum()
        df[col] = df.groupby('Libelle')[col].transform(
            lambda x: x.ffill().bfill()  # ffill puis bfill pour les premiers nulls
        )
        apres = df[col].isnull().sum()
        print(f'[FFILL] {col:35s} : {avant} nulls -> {apres} nulls restants')

print()

# GROUPE 2 — DIVIDENDES
# Remplissage par 0
# Logique : si le dividende est null, cela signifie simplement
# que la societe n a pas distribue de dividende cette annee.
# Un null ici = 0 MAD distribue — ce n est pas une donnee manquante
# mais une information metier : pas de distribution.
cols_dividende = [
    'Dernier Dividende ajusté',
    'Annee_Dividende',
    'Date_Dividende',
]

for col in cols_dividende:
    if col in df.columns:
        avant = df[col].isnull().sum()
        if col == 'Annee_Dividende':
            df[col] = df[col].fillna(0).astype(str).replace('0', 'N/A')
        elif col == 'Date_Dividende':
            df[col] = df[col].fillna('N/A')
        else:
            df[col] = df[col].fillna(0)
        apres = df[col].isnull().sum()
        print(f'[ZERO/NA] {col:33s} : {avant} nulls -> {apres} nulls restants')

print()

# GROUPE 3 — COURS DU JOUR, HAUT, BAS
# On laisse NaN intentionnellement
# Logique : un cours null signifie que l action n a pas ete
# echangee ce jour la (Statut = NT = Non Traite).
# Mettre une valeur fictive (ex: 0 ou le dernier cours) fausserait :
#   - le calcul de la volatilite (variation artificielle)
#   - le calcul du rendement journalier
#   - le Max Drawdown
# On garde NaN et on filtre ces lignes lors des calculs de KPIs.
cols_cours_laisser = ['Cours du jour', 'Cours_Haut', 'Cours_Bas']
for col in cols_cours_laisser:
    if col in df.columns:
        nb = df[col].isnull().sum()
        print(f'[NaN CONSERVE] {col:28s} : {nb} nulls conserves (action non traitee ce jour)')

print()

# GROUPE 4 — STATUT
# Remplissage par 'NT' (Non Traite)
# Logique : si le statut est null ET le cours est null,
# c est logiquement une seance sans transaction = Non Traite.
# On ne peut pas laisser le statut vide car c est une colonne
# categorielle utilisee pour filtrer les donnees dans les KPIs.
avant = df['Statut'].isnull().sum()

df['Statut'] = df.apply(
    lambda row: 'NT' if pd.isnull(row['Statut']) and pd.isnull(row['Cours du jour'])
                else ('NT' if pd.isnull(row['Statut']) else row['Statut']),
    axis=1
)

apres = df['Statut'].isnull().sum()
print(f'[NT] {"Statut":35s} : {avant} nulls -> {apres} nulls restants')
print()

# RAPPORT FINAL
print('RAPPORT FINAL')
print(f'Shape finale         : {df.shape}')
print(f'Nulls restants total : {df.isnull().sum().sum()}')
print()
print('Detail nulls restants :')
nulls_restants = df.isnull().sum()
print(nulls_restants[nulls_restants > 0].to_string())



Shape apres nettoyage basique : (46599, 22)
Nulls avant traitement        : 31733

Libelles extraits depuis Instruments : 1081
[FFILL] Capital en titres                   : 11 nulls -> 14 nulls restants
[FFILL] Val. Nom.                           : 9 nulls -> 14 nulls restants


C:\Users\poste\AppData\Local\Temp\ipykernel_2164\1981469722.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()  # ffill puis bfill pour les premiers nulls


[FFILL] Secteur activité                    : 11 nulls -> 14 nulls restants


C:\Users\poste\AppData\Local\Temp\ipykernel_2164\1981469722.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()  # ffill puis bfill pour les premiers nulls


[FFILL] Cycle de négociation                : 11 nulls -> 14 nulls restants
[FFILL] Instruments                         : 6 nulls -> 6 nulls restants
[FFILL] Libelle                             : 6 nulls -> 6 nulls restants
[FFILL] Cours de référence                  : 14 nulls -> 14 nulls restants


C:\Users\poste\AppData\Local\Temp\ipykernel_2164\1981469722.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()  # ffill puis bfill pour les premiers nulls


[FFILL] Date_Cours_Ref                      : 14 nulls -> 14 nulls restants
[FFILL] Cours_Reference_Seance              : 14 nulls -> 14 nulls restants


C:\Users\poste\AppData\Local\Temp\ipykernel_2164\1981469722.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()  # ffill puis bfill pour les premiers nulls


[FFILL] Meilleur_Demande                    : 16 nulls -> 14 nulls restants
[FFILL] Cours_Cloture                       : 1195 nulls -> 34 nulls restants
[FFILL] Plus_Haut_Janv                      : 322 nulls -> 28 nulls restants
[FFILL] Plus_Bas_Janv                       : 284 nulls -> 28 nulls restants
[FFILL] Meilleur_Offre                      : 945 nulls -> 29 nulls restants

[ZERO/NA] Dernier Dividende ajusté          : 1334 nulls -> 0 nulls restants
[ZERO/NA] Annee_Dividende                   : 639 nulls -> 0 nulls restants
[ZERO/NA] Date_Dividende                    : 1334 nulls -> 0 nulls restants

[NaN CONSERVE] Cours du jour                : 9561 nulls conserves (action non traitee ce jour)
[NaN CONSERVE] Cours_Haut                   : 5403 nulls conserves (action non traitee ce jour)
[NaN CONSERVE] Cours_Bas                    : 6180 nulls conserves (action non traitee ce jour)

[NT] Statut                              : 3343 nulls -> 0 nulls restants

RAPPORT FINAL
Shape

In [2]:
df.isnull().sum()

Type                           0
Capital en titres             14
Val. Nom.                     14
Secteur activité              14
Cycle de négociation          14
Dernier Dividende ajusté       0
Annee_Dividende                0
Date_Dividende                 0
Cours de référence            14
Date_Cours_Ref                14
Cours_Reference_Seance        14
Instruments                    6
Libelle                        6
Cours du jour               9561
Cours_Cloture                 34
Cours_Haut                  5403
Cours_Bas                   6180
Statut                         0
Meilleur_Demande              14
Meilleur_Offre                29
Plus_Haut_Janv                28
Plus_Bas_Janv                 28
dtype: int64

In [3]:
df.to_csv('Actions_CLEAN_final.csv', index=False, encoding='utf-8')
print()
print(f'Exporte : Actions_CLEAN_final.csv — {len(df):,} lignes')


Exporte : Actions_CLEAN_final.csv — 46,599 lignes
